# amltk smoke test

Smoke test for the `amltk` (AutoML Toolkit) package.

In [ ]:
# Example 1: Top-level public API surface.
import amltk

public_names = [n for n in dir(amltk) if not n.startswith('_')]
print(f"amltk public names ({len(public_names)}):")
for n in sorted(public_names)[:30]:
    print(f"  {n}: {type(getattr(amltk, n)).__name__}")

assert len(public_names) > 0, "amltk exposes no public names"

In [ ]:
# Example 2: Pipeline / Node / Component construction.
# amltk's core abstraction is a Node-based pipeline DSL.
from amltk.pipeline import Component, Sequential, Choice

# A simple identity-style component using a builtin as the item.
comp = Component(dict, name='my_dict', config={'a': 1, 'b': 2})
print(f"Component: {comp.name!r} item={comp.item!r} config={comp.config!r}")

built = comp.build_item()
print(f"built_item = {built!r}")

assert isinstance(comp, Component)
assert built == {'a': 1, 'b': 2}

In [ ]:
# Example 3: Sequential pipeline of components.
from amltk.pipeline import Component, Sequential

step1 = Component(dict, name='step1', config={'x': 1})
step2 = Component(list, name='step2', config={})

pipeline = Sequential(step1, step2, name='my_pipeline')
print(f"Sequential name = {pipeline.name!r}")
print(f"  nodes = {[n.name for n in pipeline.nodes]}")

# Iterate over the pipeline via walk() — it yields (path, node) pairs, where
# path is the list of ancestors leading to node (the root included as ([], root)).
walked = [node.name for _path, node in pipeline.walk()]
print(f"  walked = {walked}")

assert len(pipeline.nodes) == 2
assert 'step1' in walked and 'step2' in walked

In [ ]:
# Example 4: Choice node (branching in a pipeline).
from amltk.pipeline import Component, Choice

option_a = Component(dict, name='option_a')
option_b = Component(list, name='option_b')

choice = Choice(option_a, option_b, name='my_choice')
print(f"Choice name = {choice.name!r}")
print(f"  options = {[n.name for n in choice.nodes]}")

# Choice exposes its sub-nodes too.
names = {n.name for n in choice.nodes}
assert names == {'option_a', 'option_b'}

In [ ]:
# Example 5: amltk.types / amltk.randomness helpers.
# (the helper is spelled as_randomstate — no underscore — in current amltk)
import numpy as np

from amltk.randomness import as_rng, as_int, as_randomstate

rng = as_rng(42)
print(f"as_rng(42) -> {type(rng).__name__}")

seed_int = as_int(42)
print(f"as_int(42) -> {seed_int!r}")

rs = as_randomstate(42)
print(f"as_randomstate(42) -> {type(rs).__name__}")

# rng should be a numpy Generator capable of producing reproducible draws.
draw1 = as_rng(42).integers(0, 100, size=5).tolist()
draw2 = as_rng(42).integers(0, 100, size=5).tolist()
print(f"reproducible draws: {draw1} == {draw2}")

assert isinstance(rng, np.random.Generator)
assert isinstance(rs, np.random.RandomState)
assert draw1 == draw2, "as_rng must be reproducible from a fixed seed"